# Capítulo 7 · Entrenamiento y datos

Entrenar con método: tres conjuntos de datos, contaminación, optimizadores, tasa de aprendizaje, puntos de control y registro. PyTorch en CPU. El cuaderno completo tarda unos diez minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Datos, configuraciones y lotes

In [1]:
"""Capítulo 7: entrenar con método. PyTorch en CPU."""
import json
import math
import platform
import random
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

RAIZ = Path(".")
TEXTO = RAIZ / "material" / "quijote.txt"
if not TEXTO.exists():  # en Colab, descargamos la novela del repositorio
    import urllib.request
    TEXTO.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quijote.txt", TEXTO)
SALIDAS = Path("salidas")
SEMILLA = 1337
LABORATORIO = dict(contexto=32, dimension=32, cabezas=2, capas=2, lote=32)
FINAL = dict(contexto=64, dimension=64, cabezas=4, capas=4, lote=32)


# --- Datos -----------------------------------------------------------------

def preparar(texto):
    simbolos = sorted(set(texto))
    a_id = {c: i for i, c in enumerate(simbolos)}
    datos = torch.tensor([a_id[c] for c in texto], dtype=torch.long)
    a, b = int(0.8 * len(datos)), int(0.9 * len(datos))
    partes = {"entrenamiento": datos[:a], "validacion": datos[a:b],
              "prueba": datos[b:]}
    return simbolos, a_id, partes


def solapamiento(texto_a, texto_b, n=50, muestras=2000, semilla=0):
    """Cuenta los trozos de n caracteres de texto_a que están en texto_b."""
    azar = random.Random(semilla)
    inicios = [azar.randrange(len(texto_a) - n) for _ in range(muestras)]
    return sum(texto_a[i:i + n] in texto_b for i in inicios)


def lote(datos, config, generador):
    T, B = config["contexto"], config["lote"]
    inicios = torch.randint(len(datos) - T - 1, (B,), generator=generador)
    x = torch.stack([datos[i:i + T] for i in inicios])
    y = torch.stack([datos[i + 1:i + T + 1] for i in inicios])
    return x, y

In [2]:
torch.set_num_threads(4)
texto = TEXTO.read_text(encoding="utf-8")
simbolos, a_id, partes = preparar(texto)
print({nombre: len(datos) for nombre, datos in partes.items()})
corte_a, corte_b = int(0.8 * len(texto)), int(0.9 * len(texto))
for n in (5, 10, 20, 30, 50, 100):
    repetidos = solapamiento(texto[corte_b:], texto[:corte_a], n)
    print("Trozos de prueba de %d caracteres en entrenamiento: %d de 2000" % (n, repetidos))

{'entrenamiento': 1688237, 'validacion': 211030, 'prueba': 211030}
Trozos de prueba de 5 caracteres en entrenamiento: 1935 de 2000
Trozos de prueba de 10 caracteres en entrenamiento: 922 de 2000
Trozos de prueba de 20 caracteres en entrenamiento: 77 de 2000
Trozos de prueba de 30 caracteres en entrenamiento: 13 de 2000
Trozos de prueba de 50 caracteres en entrenamiento: 1 de 2000
Trozos de prueba de 100 caracteres en entrenamiento: 0 de 2000


## 2. El modelo del capítulo 6, con su tamaño como parámetro, y las dos medidas de la pérdida

In [3]:
# --- Modelo: las piezas del capítulo 6, con su tamaño como parámetro ------

class Cabeza(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["contexto"]
        tamano = C // config["cabezas"]
        self.consulta = nn.Linear(C, tamano, bias=False)
        self.clave = nn.Linear(C, tamano, bias=False)
        self.valor = nn.Linear(C, tamano, bias=False)
        self.register_buffer("mascara", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        return F.softmax(puntuaciones, dim=-1) @ v


class Bloque(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.cabezas = nn.ModuleList(
            [Cabeza(config) for _ in range(config["cabezas"])])
        self.proyeccion = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norma1, self.norma2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        return x + self.mlp(self.norma2(x))


class GPT(nn.Module):
    def __init__(self, vocabulario, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.embedding_simbolos = nn.Embedding(vocabulario, C)
        self.embedding_posiciones = nn.Embedding(config["contexto"], C)
        self.bloques = nn.Sequential(
            *[Bloque(config) for _ in range(config["capas"])])
        self.norma = nn.LayerNorm(C)
        self.salida = nn.Linear(C, vocabulario)

    def forward(self, ids):
        posiciones = torch.arange(ids.shape[1])
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))


def perdida_media(modelo, x, y):
    logits = modelo(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def evaluar(modelo, datos, lotes=20):
    """Medida rápida: la pérdida media en 20 lotes, siempre los mismos."""
    generador = torch.Generator().manual_seed(0)
    perdidas = [perdida_media(modelo, *lote(datos, modelo.config, generador))
                for _ in range(lotes)]
    return sum(p.item() for p in perdidas) / lotes


@torch.no_grad()
def evaluar_todo(modelo, datos, grupo=256):
    """Medida completa: la pérdida media en todo el bloque, trozo a trozo."""
    T = modelo.config["contexto"]
    n = (len(datos) - 1) // T
    x, y = datos[:n * T].view(n, T), datos[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, grupo):
        total += perdida_media(modelo, x[i:i + grupo],
                               y[i:i + grupo]).item() * len(x[i:i + grupo])
    return total / n

## 3. Optimizadores y calendario de la tasa

In [4]:
# --- Optimizadores y tasa de aprendizaje -------------------------------------

def crear_optimizador(modelo, nombre, tasa):
    if nombre == "sgd":
        return torch.optim.SGD(modelo.parameters(), lr=tasa)
    if nombre == "momento":
        return torch.optim.SGD(modelo.parameters(), lr=tasa, momentum=0.9)
    return torch.optim.AdamW(modelo.parameters(), lr=tasa, weight_decay=0.1)


def tasa_en(paso, tasa, pasos, calentamiento=100, final=0.1):
    """Calentamiento lineal y descenso en coseno hasta final * tasa."""
    if paso < calentamiento:
        return tasa * (paso + 1) / calentamiento
    avance = (paso - calentamiento) / max(1, pasos - calentamiento)
    return tasa * (final + (1 - final) * 0.5 * (1 + math.cos(math.pi * avance)))

In [5]:
for paso in (0, 50, 100, 1550, 2999):
    print(paso, tasa_en(paso, 3e-3, 3000))

0 3e-05
50 0.00153
100 0.003
1550 0.0016500000000000002
2999 0.0003000007921500975


## 4. Entrenar, guardar y cargar

In [6]:
# --- Entrenar, guardar y reanudar --------------------------------------------

def nuevo_estado(vocabulario, config, optimizador, tasa):
    torch.manual_seed(SEMILLA)
    modelo = GPT(vocabulario, config)
    return {"modelo": modelo,
            "optimizador": crear_optimizador(modelo, optimizador, tasa),
            "generador": torch.Generator().manual_seed(SEMILLA),
            "paso": 0, "historial": []}


def entrenar(estado, partes, hasta, tasa, pasos_totales, calendario=False,
             intervalo=250, carpeta=None):
    modelo, optimizador = estado["modelo"], estado["optimizador"]
    historial = estado["historial"]
    mejor = min([h[2] for h in historial], default=math.inf)
    while estado["paso"] <= hasta:
        paso = estado["paso"]
        ya_medido = historial and historial[-1][0] == paso
        if paso % intervalo == 0 and not ya_medido:
            medida = (paso, evaluar(modelo, partes["entrenamiento"]),
                      evaluar(modelo, partes["validacion"]))
            historial.append(medida)
            if carpeta and medida[2] < mejor:
                mejor = medida[2]
                guardar(estado, carpeta / "mejor.pt")
        if paso == hasta:
            break
        if calendario:
            for grupo in optimizador.param_groups:
                grupo["lr"] = tasa_en(paso, tasa, pasos_totales)
        x, y = lote(partes["entrenamiento"], modelo.config, estado["generador"])
        perdida = perdida_media(modelo, x, y)
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
        estado["paso"] += 1
    return estado


def guardar(estado, ruta):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"modelo": estado["modelo"].state_dict(),
                "config": estado["modelo"].config,
                "optimizador": estado["optimizador"].state_dict(),
                "generador": estado["generador"].get_state(),
                "paso": estado["paso"],
                "historial": estado["historial"]}, ruta)


def cargar(ruta, vocabulario, optimizador, tasa):
    punto = torch.load(ruta)
    estado = nuevo_estado(vocabulario, punto["config"], optimizador, tasa)
    estado["modelo"].load_state_dict(punto["modelo"])
    estado["optimizador"].load_state_dict(punto["optimizador"])
    estado["generador"].set_state(punto["generador"])
    estado["paso"], estado["historial"] = punto["paso"], punto["historial"]
    return estado

## 5. El laboratorio: optimizadores y tasas

Nueve entrenamientos de 1000 pasos con el modelo pequeño. Tarda unos tres minutos.

In [7]:
# --- Los experimentos del capítulo -------------------------------------------

def laboratorio(simbolos, partes, pasos=1000):
    resultados = {}
    pruebas = [("sgd", 0.1), ("sgd", 1.0), ("momento", 0.1),
               ("adamw", 3e-4), ("adamw", 1e-3), ("adamw", 3e-3),
               ("adamw", 1e-2), ("adamw", 3e-2), ("adamw", 1e-1)]
    for nombre, tasa in pruebas:
        estado = nuevo_estado(len(simbolos), LABORATORIO, nombre, tasa)
        entrenar(estado, partes, pasos, tasa, pasos, intervalo=100)
        resultados[f"{nombre} {tasa:g}"] = estado["historial"]
        print("%-14s validación final %.3f" % (f"{nombre} {tasa:g}",
                                               estado["historial"][-1][2]))
    SALIDAS.mkdir(parents=True, exist_ok=True)
    (SALIDAS / "laboratorio.json").write_text(
        json.dumps(resultados, indent=2), encoding="utf-8")
    return resultados

In [8]:
resultados = laboratorio(simbolos, partes)

sgd 0.1        validación final 2.432
sgd 1          validación final 2.183
momento 0.1    validación final 2.145
adamw 0.0003   validación final 2.374
adamw 0.001    validación final 2.194
adamw 0.003    validación final 1.979
adamw 0.01     validación final 1.929
adamw 0.03     validación final 1.982
adamw 0.1      validación final 2.414


## 6. Interrumpir y reanudar

In [9]:
def reanudar(simbolos, partes, pasos=600):
    seguido = nuevo_estado(len(simbolos), LABORATORIO, "adamw", 1e-2)
    entrenar(seguido, partes, pasos, 1e-2, pasos, calendario=True,
             intervalo=100)
    mitad = nuevo_estado(len(simbolos), LABORATORIO, "adamw", 1e-2)
    entrenar(mitad, partes, pasos // 2, 1e-2, pasos, calendario=True,
             intervalo=100)
    guardar(mitad, SALIDAS / "a_medias.pt")
    del mitad  # simulamos que el programa se interrumpe
    retomado = cargar(SALIDAS / "a_medias.pt", len(simbolos), "adamw", 1e-2)
    entrenar(retomado, partes, pasos, 1e-2, pasos, calendario=True,
             intervalo=100)
    return seguido["historial"][-1], retomado["historial"][-1]

In [10]:
print(reanudar(simbolos, partes))

((600, 2.0312296867370607, 2.024327254295349), (600, 2.0312296867370607, 2.024327254295349))


## 7. El experimento final

El modelo del capítulo 6 con calendario y puntos de control. Tarda unos cinco minutos y medio.

In [11]:
def experimento_final(simbolos, partes, pasos=3000, tasa=3e-3):
    estado = nuevo_estado(len(simbolos), FINAL, "adamw", tasa)
    inicio = time.perf_counter()
    entrenar(estado, partes, pasos, tasa, pasos, calendario=True,
             carpeta=SALIDAS)
    segundos = time.perf_counter() - inicio
    mejor = cargar(SALIDAS / "mejor.pt", len(simbolos), "adamw", tasa)
    modelo = mejor["modelo"]
    registro = {
        "fecha": time.strftime("%Y-%m-%d %H:%M"),
        "python": platform.python_version(), "torch": torch.__version__,
        "hilos": torch.get_num_threads(), "semilla": SEMILLA,
        "division": "80/10/10 en bloques seguidos", "config": FINAL,
        "parametros": sum(p.numel() for p in modelo.parameters()),
        "optimizador": "adamw", "freno": 0.1, "tasa": tasa,
        "calendario": "calentamiento de 100 pasos y coseno hasta el 10 %",
        "pasos": pasos, "segundos": round(segundos),
        "medidas": "20 lotes cada 250 pasos; bloques completos al final",
        "historial": estado["historial"], "mejor_paso": mejor["paso"],
        "validacion": evaluar_todo(modelo, partes["validacion"]),
        "prueba": evaluar_todo(modelo, partes["prueba"])}  # una sola vez
    (SALIDAS / "registro.json").write_text(
        json.dumps(registro, indent=2, ensure_ascii=False), encoding="utf-8")
    torch.save({"modelo": modelo.state_dict(), "config": FINAL,
                "simbolos": simbolos}, SALIDAS / "modelo_final.pt")
    return registro

In [12]:
registro = experimento_final(simbolos, partes)
print("Mejor paso:", registro["mejor_paso"], "· validación: %.3f" % registro["validacion"],
      "· prueba: %.3f" % registro["prueba"])
print((SALIDAS / "registro.json").read_text(encoding="utf-8")[:900])

Mejor paso: 3000 · validación: 1.478 · prueba: 1.474
{
  "fecha": "2026-10-03 13:51",
  "python": "3.12.14",
  "torch": "2.8.0+cpu",
  "hilos": 4,
  "semilla": 1337,
  "division": "80/10/10 en bloques seguidos",
  "config": {
    "contexto": 64,
    "dimension": 64,
    "cabezas": 4,
    "capas": 4,
    "lote": 32
  },
  "parametros": 215260,
  "optimizador": "adamw",
  "freno": 0.1,
  "tasa": 0.003,
  "calendario": "calentamiento de 100 pasos y coseno hasta el 10 %",
  "pasos": 3000,
  "segundos": 325,
  "medidas": "20 lotes cada 250 pasos; bloques completos al final",
  "historial": [
    [
      0,
      4.614071202278137,
      4.615765905380249
    ],
    [
      250,
      2.211728882789612,
      2.2218462109565733
    ],
    [
      500,
      1.9435224890708924,
      1.9617477893829345
    ],
    [
      750,
      1.7880217671394347,
      1.8076424241065978
    ],
    [
      1000,
      1.7041339337825776,
      1.729242384433


## Ejercicios

1. Explica con tus palabras para qué sirve cada uno de los tres conjuntos de datos. ¿Qué ocurriría si eligiéramos la tasa de aprendizaje mirando la pérdida de prueba?

2. ¿Por qué cortamos el texto en bloques seguidos en lugar de repartir frases o fragmentos al azar?

3. Según la tabla del capítulo, encontrar en el entrenamiento un trozo de 10 caracteres de la prueba no indica contaminación, pero encontrar uno de 100 sí. ¿Por qué?

4. Con el modelo de laboratorio, cada lote tiene 32 fragmentos de 32 caracteres. ¿Cuántas predicciones hace en 1000 pasos? ¿Cuántas épocas son?

5. Aplica dos pasos del momento a mano, con v=0 al principio, tasa 0,1 y gradientes g_1=1 y g_2=1. ¿Cuánto se mueve el parámetro en total? Compáralo con dos pasos de descenso de gradiente simple.

6. Un parámetro tiene siempre un gradiente de 0,001 y otro, de 10. ¿Cuánto se mueve cada uno en un paso de descenso de gradiente con tasa 0,003? ¿Y aproximadamente con Adam, cuando m y s ya se han estabilizado?

7. Con la función `tasa_en` del programa, calcula la tasa en los pasos 0, 50, 100, 1550 y 2999 del entrenamiento final.

8. Abre `salidas/laboratorio.json` y busca, para cada tasa de AdamW, la pérdida de validación en el paso 300. ¿Qué tasa gana a los 300 pasos? ¿Y a los 1000? ¿Desde qué paso gana ya siempre la segunda? ¿Qué te enseña esto sobre la duración de un barrido?

9. Comprueba qué pasa si olvidamos el estado del optimizador. Quita de `cargar` la línea que lo carga, ejecuta solo la prueba de reanudación (en el cuaderno, su celda) y compara las dos pérdidas. ¿Por qué ya no coinciden? ¿Por qué la diferencia es tan pequeña?

10. Vuelve a dejar `cargar` como estaba. Después modifica `guardar` para que no guarde el estado del generador, quita de `cargar` la línea con `set_state` y repite la prueba de reanudación. Explica el resultado.

11. Barrido con el modelo final. Adapta `laboratorio` para que use la configuración `FINAL` y solo AdamW con las tasas 0,001, 0,003 y 0,01, y compara su pérdida de validación tras 1000 pasos. Tarda unos seis minutos. ¿Sigue siendo 0,003 una buena elección? ¿Prefiere el modelo grande una tasa menor que el de laboratorio?

12. Registro de entrenamiento. Abre `salidas/registro.json` y escribe un informe breve, de unas diez líneas, con la configuración, la evolución de las pérdidas, el mejor punto de control y las pérdidas finales. Indica qué decisiones se tomaron con la validación y cuándo se usó la prueba.

13. Introduce a propósito una contaminación: añade al final del texto de entrenamiento los primeros 50 000 caracteres del conjunto de prueba y vuelve a medir el solapamiento con trozos de 50 caracteres. ¿Qué esperarías que pasara con la pérdida de prueba si entrenaras así?

## Referencias

Inspirado en [nanoGPT](https://github.com/karpathy/nanoGPT) y en la lección [Let's reproduce GPT-2](https://www.youtube.com/watch?v=l8pRSuU81PU) de Andrej Karpathy. Texto de *Don Quijote de la Mancha*, de dominio público. Código y experimentos propios.